# 02 - Feature Engineering

### Objective

Transform cleaned NBA player statistics into interpretable measures of playing style.

### Goals

1. Combine validated datasets.
2. Normalize statistical activity.
3. Engineer offensive and defensive style metrics.
4. Evaluate feature quality and redundancy.

### Modeling Philosophy

Playing style and player performance are treated separately.

The style representation of a player focuses on tendencies, actions, usage, responsibilities, and shot selection. Efficiency and performance outcomes are reserved for later player evaluation.

Per 100 possessions is the preferred normalization unit for this analysis because it not only adjusts for playing time differences but also controls for pace and game speed which can vary by team and era.

### Inputs

- Validated NBA datasets

### Outputs

- Player-level engineered statistics
- Playing-style dimensions
- Feature dictionary

In [ ]:
import pandas as pd
import numpy as np
import duckdb

from nba_archetypes.config import (PROJECT_ROOT, RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, SQL_DIR, MIN_MINUTES)
from nba_archetypes.data_validation import validate_candidate_features
from nba_archetypes.feature_engineering import safe_divide, per_100_calc

# Load eligible players dataset.
eligible_players = pd.read_parquet(INTERIM_DATA_DIR / "eligible_players.parquet")
eligible_ids = set(eligible_players["PLAYER_ID"])

# Load existing raw datasets.
datasets = {file.stem: pd.read_parquet(file) for file in RAW_DATA_DIR.glob("*.parquet")}

### 1. Feature Source Verification

Before engineering our offensive and defensive features, we verify that the necessary raw columns exist, inspect their values, and document the planned calculations.

We will also be documenting whether there are any missing eligible players and we need to inspect whether the minimum, median, and maximum values look plausible for that statistic.

In [20]:
required_columns = [
    # Statistics needed for feature engineering calculations
    "POSS",
    "FGA",

    # Offensive features
    "FG3A",
    "USG_PCT",
    "TOUCHES",
    "AVG_SEC_PER_TOUCH",
    "POTENTIAL_AST",
    "PASSES_MADE",
    "DRIVES",
    "RA_FGA",
    "PULL_UP_FGA",
    "PCT_UAST_FGM",
    "CATCH_SHOOT_FGA",
    "PAINT_TOUCHES",
    "POST_TOUCHES",
    "SCREEN_ASSISTS",
    "OREB_CHANCES",

    #Defensive features
    "DEFLECTIONS",
    "STL",
    "BLK",
    "CONTESTED_SHOTS_3PT",
    "CONTESTED_SHOTS_2PT",
    "DREB_CHANCES"
]

results = []

for column in required_columns:
    found = False
    for dataset_name, df in datasets.items():
        if dataset_name.startswith("playtype_") or dataset_name == "matchup_stats":
            continue
        if column not in df.columns:
            continue
        found = True
        eligible_data = df[df["PLAYER_ID"].isin(eligible_ids)]
        values = pd.to_numeric(eligible_data[column], errors="coerce")

        results.append({
                "column": column,
                "dataset": dataset_name,
                "status": "Found",
                "players": eligible_data["PLAYER_ID"].nunique(),
                "missing": values.isna().sum(),
                "minimum": values.min(),
                "median": values.median(),
                "maximum": values.max()
        })
    if not found:
        results.append({
            "column": column,
            "dataset": None,
            "status": "Missing",
            "players": None,
            "missing": None,
            "minimum": None,
            "median": None,
            "maximum": None
        })

column_report = pd.DataFrame(results)

display(column_report)

multiple_sources = column_report[column_report.duplicated("column", keep=False)]
display(multiple_sources)

,column,dataset,status,players,missing,minimum,median,maximum
0,POSS,advanced_stats,Found,450,0,537.000,2588.500,6040.000
1,FGA,advanced_stats,Found,450,0,60.000,403.000,1543.000
2,FGA,base_stats,Found,450,0,60.000,403.000,1543.000
3,FGA,scoring_stats,Found,450,0,60.000,403.000,1543.000
4,FG3A,base_stats,Found,450,0,0.000,165.500,740.000
5,USG_PCT,advanced_stats,Found,450,0,0.082,0.177,0.368
6,USG_PCT,usage_stats,Found,450,0,0.082,0.177,0.368
7,TOUCHES,painttouch_stats,Found,450,0,234.000,1755.500,6134.000
8,TOUCHES,possessions_stats,Found,450,0,234.000,1755.500,6134.000
9,TOUCHES,posttouch_stats,Found,450,0,234.000,1755.500,6134.000


,column,dataset,status,players,missing,minimum,median,maximum
1,FGA,advanced_stats,Found,450,0,60.000,403.000,1543.000
2,FGA,base_stats,Found,450,0,60.000,403.000,1543.000
3,FGA,scoring_stats,Found,450,0,60.000,403.000,1543.000
5,USG_PCT,advanced_stats,Found,450,0,0.082,0.177,0.368
6,USG_PCT,usage_stats,Found,450,0,0.082,0.177,0.368
7,TOUCHES,painttouch_stats,Found,450,0,234.000,1755.500,6134.000
8,TOUCHES,possessions_stats,Found,450,0,234.000,1755.500,6134.000
9,TOUCHES,posttouch_stats,Found,450,0,234.000,1755.500,6134.000
18,PAINT_TOUCHES,painttouch_stats,Found,450,0,3.000,85.500,823.000
19,PAINT_TOUCHES,possessions_stats,Found,450,0,3.000,85.500,823.000


### 2. Denominator Validation

Rather than feeding the raw totals of the stats we collected above into our model, we are going to convert the raw totals into per 100 possessions or share of field goal attempts with our feature engineering calculations. This is to ensure that our model doesn't improperly weight players based off volume of playing time or pace of play relative to their team.

The denominators for these feature engineering calculations are total possession and total field goal attempts per player so we need to make sure these denominators are statistically valid for our calculations.

We are calculating the approximate possessions per 48 minutes for each player which should land somewhere near 100 on average for statistically valid possession counts. We are also making sure that any field goal attempt shares that we are calculating have less than the total number of field goal attempts for any given player.

In [14]:
base = datasets["base_stats"]
advanced = datasets["advanced_stats"]

exposure = (
    base[["PLAYER_ID", "PLAYER_NAME", "MIN"]]
    .merge(
        advanced[["PLAYER_ID", "MIN", "POSS", "GP"]],
        on="PLAYER_ID",
        suffixes=("_base", "_advanced"),
        validate="one_to_one",
    )
)

exposure = exposure[
    exposure["PLAYER_ID"].isin(eligible_ids)
].copy()

# Check consistency between source minutes.
exposure["minute_difference"] = (
    exposure["MIN_base"] - (exposure["MIN_advanced"] * exposure["GP"])
).abs()

# Approximate possessions per 48 minutes.
exposure["poss_per_48"] = (
    exposure["POSS"] / exposure["MIN_base"] * 48
)

print("Players checked:", len(exposure))
print("Maximum minute difference:", exposure["minute_difference"].max())

display(exposure["poss_per_48"].describe())

Players checked: 450
Maximum minute difference: 3.8733333333334485


count    450.000000
mean     101.659499
std        2.481528
min       95.329092
25%      100.092529
50%      101.699729
75%      103.329634
max      110.245479
Name: poss_per_48, dtype: float64

In [24]:
shot_check = (
    datasets["base_stats"][
        ["PLAYER_ID", "FGA", "FG3A"]
    ]
    .merge(
        datasets["shot_locations"][
            ["PLAYER_ID", "RA_FGA"]
        ],
        on="PLAYER_ID",
        validate="one_to_one",
    )
    .merge(
        datasets["pullupshot_stats"][
            ["PLAYER_ID", "PULL_UP_FGA"]
        ],
        on="PLAYER_ID",
        validate="one_to_one",
    )
)

shot_check = shot_check[
    shot_check["PLAYER_ID"].isin(eligible_ids)
]

checks = {
    "3PA > total FGA":
        (shot_check["FG3A"] > shot_check["FGA"]).sum(),

    "RA FGA > total FGA":
        (shot_check["RA_FGA"] > shot_check["FGA"]).sum(),

    "Pull-up FGA > total FGA":
        (shot_check["PULL_UP_FGA"] > shot_check["FGA"]).sum()
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["check", "violations"]
    )
)

,check,violations
0,3PA > total FGA,0
1,RA FGA > total FGA,0
2,Pull-up FGA > total FGA,0


### 3. Matchup Data Duplicate Check

Checking to see if there are any defender-opponent duplicate pairs.

In [22]:
matchups = datasets["matchup_stats"]

print("Matchup rows:", len(matchups))

print(
    "Duplicate defender-opponent pairs:",
    matchups.duplicated(
        ["DEF_PLAYER_ID", "OFF_PLAYER_ID"]
    ).sum()
)

Matchup rows: 147805
Duplicate defender-opponent pairs: 0


In [30]:
source_names = [
    "base_stats",
    "advanced_stats",
    "possessions_stats",
    "passing_stats",
    "drives_stats",
    "shot_locations",
    "pullupshot_stats",
    "scoring_stats",
    "catchshoot_stats",
    "painttouch_stats",
    "posttouch_stats",
    "hustle_stats",
    "rebounding_stats",
]

source_paths = {name: RAW_DATA_DIR / f"{name}.parquet" for name in source_names}

source_paths["eligible_players"] = (INTERIM_DATA_DIR / "eligible_players.parquet")

missing_files = [
    name for name, path in source_paths.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing Parquet files: {missing_files}")

sql_query = (SQL_DIR / "02_style_features.sql").read_text()

with duckdb.connect() as con:
    for view_name, file_path in source_paths.items():
        con.execute(f"""
            CREATE OR REPLACE VIEW {view_name} AS
            SELECT *
            FROM read_parquet('{file_path.as_posix()}')
        """)
    style_features = con.execute(sql_query).df()

display(style_features.head())

,PLAYER_ID,PLAYER_NAME,POSS,FGA,FG3A,STL,BLK,USG_PCT,TOUCHES,AVG_SEC_PER_TOUCH,...,PCT_UAST_FGM,CATCH_SHOOT_FGA,PAINT_TOUCHES,POST_TOUCHES,SCREEN_ASSISTS,DEFLECTIONS,CONTESTED_SHOTS_3PT,CONTESTED_SHOTS_2PT,OREB_CHANCES,DREB_CHANCES
0,1631260,AJ Green,4704,618,554,39,8,0.135,2930,3.46,...,0.099,417,37,0,28,111,134,172,66,286
1,1642358,AJ Johnson,968,176,57,11,3,0.195,676,4.97,...,0.561,36,27,1,3,22,30,34,26,58
2,203932,Aaron Gordon,2096,398,157,21,10,0.213,1582,4.11,...,0.293,108,191,68,36,27,35,67,80,197
3,1628988,Aaron Holiday,1618,252,170,27,5,0.159,1041,5.00,...,0.314,145,6,4,6,39,53,49,22,65
4,1630174,Aaron Nesmith,2871,512,277,26,23,0.190,2097,3.14,...,0.113,226,51,11,27,50,100,150,86,192


In [35]:
missing_report = (
    style_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_report[missing_report > 0]
)

validate_candidate_features(style_features, eligible_players)

Series([], dtype: int64)

KeyError: 'MIN'